<a href="https://colab.research.google.com/github/kj824/PyTorch-Basic-Study/blob/main/PyTorch%EA%B8%B0%EC%B4%88_%EB%AA%A8%EB%8D%B8%ED%95%99%EC%8A%B5%EB%B0%8F%EC%B6%94%EB%A1%A0(260313).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pytorch
- Python 기반 과학 컴퓨팅 패키지
- GPU 및 기타 가속기의 성능을 활용하기 위해 Numpy를 대체하는 라이브러리
- 신경망 구현에 유용한 자동 미분 라이브러리

## 데이터 작업하기
라이브러리 불러오기

In [6]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

In [7]:
# FashionMNIST 데이터셋 사용
training_data = datasets.FashionMNIST(
    root="data", # data라는 폴더에 데이터를 저장
    train=True, # 학습데이터
    download=True, # 폴더에 데이터가 없다면 인터넷에서 다운로드
    transform=ToTensor(), # 이미지를 파이토치가 계산할 수 있는 숫자(Tensor)로 변경
)

test_data = datasets.FashionMNIST(
    root="data",
    train=False, # 테스트 데이터
    download=True,
    transform=ToTensor(),
)

In [8]:
print(len(training_data))
print(len(test_data))

60000
10000


In [9]:
batch_size = 64

# Create data loaders.
train_dataloader = DataLoader(training_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

for X, y in test_dataloader: # X: 이미지 데이터(문제), y : 정답 레이블(정답)
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break # 설정한 64개만 확인하고 멈추기

Shape of X [N, C, H, W]: torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64]) torch.int64


`torch.Size([64, 1, 28, 28])`
- 64 : 한 묶음에 들어있는 이미지 개수
- 1 : 색상 채널 (1은 흑백, 3은 컬러)
- 28, 28 : 이미지의 가로, 세로 픽셀 크기

## 모델 생성
PyTorch에서 신경망을 정의하기 위해 'nn.Module' 을 상속하는 클래스 생성

In [10]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

# Define model
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()

        # 이미지를 1차원 데이터로 (28x28 행렬 -> 784개의 긴 줄)
        self.flatten = nn.Flatten()

        # 신경망 레이어 차례대로 쌓기
        self.linear_relu_stack = nn.Sequential(
            # 첫번째 층: 784개의 입력을 받아 512개의 특징으로 확장
            nn.Linear(28*28, 512),

            # 활성화 함수: 복잡한 패턴을 학습할 수 있게 도와주는 역할
            nn.ReLU(),

            # 두번째 층: 512개의 특징을 다시 512개의 특징으로 연결
            nn.Linear(512, 512),
            nn.ReLU(),

            # 마지막 층: 최종적으로 10가지 카테고리로 결과 도출
            nn.Linear(512, 10)
        )

    def forward(self, x):
        # 데이터가 모델에 들어왔을 때 방향 정의
        # 데이터를 편다(flatten)
        x = self.flatten(x)
        # 쌓아둔 층을 통과시켜 결과(logits)를 얻는다
        logits = self.linear_relu_stack(x)
        return logits

# 모델 객체를 생성하고 결정된 장치(GPU/CPU)로 보낸다
model = NeuralNetwork().to(device)
print(model)

Using cpu device
NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


- CPU 사용
- (0) Linear: 784 -> 512  
입력된 784개의 픽셀 정보를 512개의 특징으로 복잡하게 조합
- (1) ReLU: 활성화 함수  
계산된 값 중 필요없는(0이하) 값은 버리고 중요한 값만 살림 (필터링)
- (2) Linear: 512 -> 512  
추출된 특징들을 한 번 더 가공해서 더 깊은 의미를 찾음
- (3) ReLU: 다시 한번 중요한 정보만 필터링
- (4) Linear: 512 -> 10  
중요! 모든 정보를 취합해 10개의 숫자로 최종 요약

## 모델 매개변수 최적화
모델을 학습시키기 위해 손실 함수와 최적화 도구가 필요
- 손실 함수: 모델의 출력값(Logits)과 실제 정답(Target) 사이의 정보 entropy 차이를 계산하는 지표
- 최적화 알고리즘: 손실 함수를 통해 계산된 기울기(Gradient)를 사용하여 모델의 매개변수를 업데이트 하는 알고리즘

In [14]:
loss_fn = nn.CrossEntropyLoss() # 손실 함수
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3) # 최적화 알고리즘
# model.parameters() : 최적화 대상
# lr=1e-3 : 학습률

In [15]:
def train(dataloader, model, loss_fn, optimizer):

    # 전체 데이터셋의 총 샘플 수를 가져옴
    size = len(dataloader.dataset)
    model.train() # 학습 모드

    # dataloader로부터 batch 단위로 데이터(X: 이미지, y: 정답)를 꺼내 반복
    for batch, (X, y) in enumerate(dataloader):
        # 데이터를 모델이 연산된 장치(CPU/GPU)로 전송
        X, y = X.to(device), y.to(device)

        # Forward Propagation (순전파)
        # 현재 모델에 입력값(X)를 넣어 예측값(pred)을 계산
        pred = model(X)

        # Compute Loss (순전파)
        # 예측값(pred)과 실제 정답(y)을 비교하여 손실 함수(loss_fn)로 오차를 수치화
        loss = loss_fn(pred, y)

        # Backpropagation (역전파)
        # 손실값에 대해 각 매개변수의 기울기 계산
        loss.backward()

        # Update Parameters (가중치 업데이트)
        # 계산된 기울기를 바탕으로 optimizer가 모델의 가중치를 수정
        optimizer.step()
        # 파이토치는 기울기를 누적하는 성질이 있음, 다음 배치를 처리하기 전 반드시 0으로 비워줌
        optimizer.zero_grad()

        # 학습 진행 상황 출력 (100번째 배치마다 현재 손실값과 처리된 데이터 수 출력)
        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

모델이 학습하고 있는지 확인하기 위해 테스트 데이터셋을 사용하여 모델의 성능 점검

In [19]:
def test(dataloader, model, loss_fn):
    # 전체 데이터셋의 크기와 배치 개수 파악
    size = len(dataloader.dataset) # 전체 테스트 데이터 개수
    num_batches = len(dataloader) # 배치 개수 (전체 데이터 / 배치 사이즈)

    # 모델을 평가 모드로 전환
    # Dropout이나 Batch Normalization 같은 층들이 학습 때와 다르게 동작하도록 설정
    model.eval()

    # 누적할 손실값(loss)과 맞춘 개수(correct)를 초기화
    test_loss, correct = 0, 0

    # 기울기 계산을 비활성화
    # 평가 시에는 역전파를 하지 않으므로 메모리 사용량을 줄이고 속도를 높임
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device) # 데이터를 저장된 장치로 보냄
            pred = model(X) # 모델에 데이터를 넣어 예측값을 얻음

            # 손실 함수를 이용해 오차를 계산하고 누적
            # .item()은 텐서에서 순수 스칼라 값을 추출
            test_loss += loss_fn(pred, y).item()

            # 정확도 계산: 예측값 중 가장 높은 확률을 가진 인덱스(argmax)와 실제 정답(y)을 비교
            # 맞으면 1, 틀리면 0으로 변환하여 모두 더한 뒤 누적
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    # 평균 손실과 최종 정확도 산출
    test_loss /= num_batches # 누적 오차 / 배치 개수 = 평균 오차
    correct /= size # 밎춘 개수 / 전체 데이터 개수 = 정확도

    # 결과 출력
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

### 궁금증1: tensor와 .item()의 관계

In [20]:
# 예시 코드
loss_tensor = torch.tensor(2.3501) # 텐서 상자
print(loss_tensor)

pure_number = loss_tensor.item() # 텐서 안의 숫자만 추출
print(pure_number)

tensor(2.3501)
2.350100040435791


`.item()`은 오직 값이 1개인 텐서(스칼라)에서만 사용 가능

### 궁금증2: 순전파와 역전파

- 순전파
    - 입력 데이터(x)가 모델의 각 층(layer)을 통과하여 가중치(W) 및 편향(b)과 연산되어 최종 예측값을 산출하는 과정
    - 입력에 대한 모델의 현재 판단 결과를 도출하고, 실제 정답과의차이인 손실(loss)을 계산하기 위함
- 역전파
    - 출력층에서 발생한 손실(loss)을 입력층 방향으로 거슬러 올라가며 각 층의 가중치와 편향에 대한 편미분 값을 계산하는 과정
    - 가중치를 어느 방향으로 얼마나 수정해야 손실이 최소화될지 파악하기 위함

### 궁금증3: 가중치와 기울기

- 가중치 (Weight)
    - 인공신경망의 각 노드 간 연결 강도를 나타내는 파라미터
    - 입력 신호가 결과 출력에 미치는 영향력을 조절하는 수치. 최적의 예측을 가능하게 하는 가중치 집합을 찾는 것
- 기울기 (Gradient)
    - 손실 함수의 값이 가장 가파르게 증가하는 방향을 가리킴. 모델은 이 기울기의 반대 방향으로 가중치를 업데이트하여 손실을 줄여나감 (경사하강법)

In [22]:
# 에폭 설정: 전체 데이터셋을 총 몇 번 반복할 것인지
epochs = 5

# 지정한 에폭 수만큼 반복문 실행
for t in range(epochs):
    # 현재 몇 번째 에폭인지 출력 (t가 0부터 시작하므로 +1을 하여 1부터 표시됨)
    print(f"Epoch {t+1}\n-------------------------------")

    # 학습  단계 (train)
    # 모델의 가중치와 편향을 업데이트하여 모델의 오차를 줄여나감
    # optimizer가 포함되어 있어 실제로 모델이 공부하는 단계
    train(train_dataloader, model, loss_fn, optimizer)

    # 검증 단계
    # 학습된 모델이 새로운 데이터를 얼마나 잘 맞히는지 성능 확인
    # 가중치를 업데이트하지 않고 오직 평가만 진행
    test(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 1.170299  [   64/60000]
loss: 1.162288  [ 6464/60000]
loss: 0.983513  [12864/60000]
loss: 1.122513  [19264/60000]
loss: 0.990786  [25664/60000]
loss: 1.030545  [32064/60000]
loss: 1.064448  [38464/60000]
loss: 0.999078  [44864/60000]
loss: 1.036225  [51264/60000]
loss: 0.983052  [57664/60000]
Test Error: 
 Accuracy: 66.0%, Avg loss: 0.988151 

Epoch 2
-------------------------------
loss: 1.049785  [   64/60000]
loss: 1.062469  [ 6464/60000]
loss: 0.866797  [12864/60000]
loss: 1.028707  [19264/60000]
loss: 0.897143  [25664/60000]
loss: 0.933117  [32064/60000]
loss: 0.982939  [38464/60000]
loss: 0.921762  [44864/60000]
loss: 0.953360  [51264/60000]
loss: 0.914110  [57664/60000]
Test Error: 
 Accuracy: 67.4%, Avg loss: 0.913509 

Epoch 3
-------------------------------
loss: 0.961508  [   64/60000]
loss: 0.993471  [ 6464/60000]
loss: 0.783036  [12864/60000]
loss: 0.962665  [19264/60000]
loss: 0.834256  [25664/60000]
loss: 0.861874  [32064/600

- loss(손실값)이 0에 가까울수록 완벽하게 맞히고 있다는 의미
    - Epoch 1~5까지의 첫 번째 줄을 봤을 때 1.17 -> 1.04 -> 0.96 -> 0.89 -> 0.83으로 역행 없이 줄어들고 있다
- 정확도(Accuracy)
    - epoch1이 66.0%, epoch5가 71.0%으로 5%의 정확도가 상승했다
- 평균 오차(Avg Loss)
    - epoch1이 0.988, epoch5가 0.786으로 약 20% 감소했다
- 알아두기
    - 학습 loss는 낮은데 Avg loss만 높다면 모델이 답만 외운 상태(과적합)일 확률이 높

## 모델 저장

내부 상태 사전(모델 매개변수 포함)을 직렬화하는 것

In [24]:
# 메모리에 있는 state_dict 객체를 하드디스크에 물리적인 파일(.pth) 형태로저장
torch.save(model.state_dict(), "model.pth")
print("Saved PyTorch Model State to model.pth")

Saved PyTorch Model State to model.pth


## 모델 로딩 중

모델을 로드하는 과정에서 모델 구조를 다시 생성하고 상태 사전을 그 안에 로드하는 작업이 포함

In [26]:
# 저장된 가중치를 입힐 빈 모델 객체를 생성하고 지정된 장치(CPU/GPU)로 보냄
model = NeuralNetwork().to(device)
# .pth 파일에서 가중치 데이터만 안전하게 로드하여(weights_only=True)
# 생성한 모델 객체의 각 파라미터(W,b)에 할당
model.load_state_dict(torch.load("model.pth", weights_only=True))

<All keys matched successfully>

모델을 이제 예측에 사용할 수 있음

In [29]:
# 모델이 예측할 클래스(정답 레이블)의 이름을 순서대로 리스트화
classes = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]

# 모델을 평가 모드로 전환
# 추론 시 불필요한 Dropout, Batch Normalization등의 기능을 비활성화
model.eval()

# 테스트 데이터셋의 첫 번째(인덱스 0) 샘플에서 입력 데이터(x)와 정답(y)을 가져옴
x, y = test_data[0][0], test_data[0][1]

# 기울기 계산을 하지 않도록 설정하여 메모리 효율을 높이고 연산 속도 가속화
with torch.no_grad():
    x = x.to(device) # 데이터를 모델이 있는 장치로 이동
    pred = model(x) # 모델에 데이터를 입력하여 출력값 산출

    # 결과 해석
    # pred[0].argmax(0) : 출력된 확률 값 중 가장 큰 값의 인덱스를 추출
    # classes[...] : 추출된 인덱스를 바탕으로 해당되는 클래스 이름을 매칭
    predicted, actual = classes[pred[0].argmax(0)], classes[y]

    # 예측 결과와 실제 정답을 비교 출력
    print(f'Predicted: "{predicted}", Actual: "{actual}"')

Predicted: "Ankle boot", Actual: "Ankle boot"


### 추가 (10개의 데이터 순회)

In [34]:
model.eval()
with torch.no_grad():
    # 0번부터 9번까지 10개의 데이터 순회
    for i in range(10):
        x, y = test_data[i][0], test_data[i][1]
        x = x.to(device)

        pred = model(x)

        predicted = classes[pred[0].argmax(0)]
        actual = classes[y]

        print(f'[{i}] Predicted: "{predicted}", Actual: "{actual}"')

[0] Predicted: "Ankle boot", Actual: "Ankle boot"
[1] Predicted: "Pullover", Actual: "Pullover"
[2] Predicted: "Trouser", Actual: "Trouser"
[3] Predicted: "Trouser", Actual: "Trouser"
[4] Predicted: "Shirt", Actual: "Shirt"
[5] Predicted: "Trouser", Actual: "Trouser"
[6] Predicted: "Shirt", Actual: "Coat"
[7] Predicted: "Coat", Actual: "Shirt"
[8] Predicted: "Sneaker", Actual: "Sandal"
[9] Predicted: "Sneaker", Actual: "Sneaker"


- 정확도: 10개 중 7개를 맞혔으므로 70.0%
- 오답 분석:
    - 6번: Predicted: "Shirt", Actual: "Coat" -> 코트를 셔츠로 오인
    - 7번: Predicted: "Coat", Actual: "Shirt" -> 셔츠를 코트로 오인
    - 8번: Predicted: "Sneaker", Actual: "Sandal" -> 샌들을 운동화로 오인
- 앞서 확인했던 전체 테스트 정확도 71.0%와 거의 일치하는 결과
- 상의나 신발처럼 시각적으로 특징이 비슷한 물체사이에서 변별력이 부족한 것으로 보임

### 궁금증4: `.eval()` 호출 시 비활성 되는 기능

- 모델을 `.eval` 모드로 전환하면 추론시 일관성과 정확도를 위해 특정 레이어의 동작 방식이 고정됨
- 드롭아웃
    - 학습 시: 과적합(Overfitting) 방지를 위해 무작위로 일부 뉴런의 출력을 0으로 만듦
    - 평가 시 비활성화 이유: 추론 시에는 모델이 가진 모든 정보를 활용해야 하며, 동일한 입력에 대해 항상 일관된 결과를 내놓아야 하기 때문
- 배치 정규화
    - 학습 시: 현재 입력된 배치 단위의 평균과 분산을 게산하여 데이터를 정규화
    - 평가 시 비활성화 이유: 평가 시에는 배치 크기가 1일 수도 있어 통계치가 부정확할 수 있다. 따라서 이동평균과 이동분산을 고정값으로 사용하여 정규화를 수행
- 평가 하는 때는 언제인가?
    - 검증(Validation): 1epoch이 끝날때마다 학습이 잘 되고 있는지 확인하기 위해
    - 추론(Inference): 실제로 모델을 서비스에 배포해서 사용자가 올린 새로운 이미지가 무엇인지 맞히게 할 때

모델을 평가할 때 가장 중요한 것은 일관성과 객관성이다.